# FAT10–MAD2 Interface — Agentic Adjudication Pipeline

An LLM-agent pipeline that **retrieves the literature claim, grounds it in real
structures/sequence, compares it against real MD evidence, and stages a multi-agent
debate** to reach an honest, calibrated verdict — run it cell by cell below.

**Honest status:** the MD data, PDBe structures, and UniProt domain boundaries are
real. The interface *conclusion* is a method demo: there is **no experimental
FAT10–MAD2 complex structure**, so nothing here is ground truth.

> Set `DEEPSEEK_API_KEY` in your environment before launching Jupyter to enable the
> LLM stages (literature extraction + debate). The non-LLM stages run without it.


## Setup


In [ ]:
import os, sys, json, re
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')                      # run from repo root
sys.path.insert(0, os.getcwd())
from IPython.display import Markdown, display
def md(x): display(Markdown(x))
HAVE_KEY = bool(os.environ.get('DEEPSEEK_API_KEY'))
md(f"**Repo root:** `{os.getcwd()}`  \n**DEEPSEEK_API_KEY set:** {HAVE_KEY}")

## Stage 1 — Retrieve the binding-region claim from the literature

An agent searches PubMed and extracts which FAT10 region binds MAD2 (falls back to a
cited result if PubMed is unreachable, and says so).


In [ ]:
if HAVE_KEY:
    from agent.literature import retrieve_binding_region
    lit = retrieve_binding_region()
else:
    lit = {'retrieved_live': False,
           'claim': 'MAD2 binds the N-terminal (first) ubiquitin-like domain of FAT10',
           'source': 'Theng et al. 2014 (no API key: live search skipped)', 'pmids': []}
tag = '🔎 live PubMed' if lit.get('retrieved_live') else '📌 cited fallback'
md(f"**Claim ({tag}):** {lit.get('claim','?')}  \n"
   f"**Source:** {lit.get('source','?')}  \n**PMIDs:** {lit.get('pmids') or '—'}")

## Stage 2 — Ground it: real structures + UniProt-verified domains


In [ ]:
from agent.tools import fetch_structures, get_expected_interface_region
st = fetch_structures('O15205')
rows = '\n'.join(f"| {s['pdb_id'].upper()} | {s.get('resolution','-')} | {s.get('experimental_method','-')} |"
                 for s in st['structures'])
md('**PDBe structures (verified FAT10):**\n\n| PDB | Resolution | Method |\n|---|---|---|\n' + rows)
exp = get_expected_interface_region('O15205')
md(f"\n**UniProt-verified domains** — UBL1 (N-term, expected MAD2 site): **{exp['residue_range']}**; "
   f"others: {exp.get('other_domains')}")

## Stage 3 — Real MD interface evidence (your 100 ns simulation)


In [ ]:
scores = json.load(open('analysis/md_interface_scores.json'))
method, occ = next(iter(scores.items()))
DOMAINS = [('UBL1 (N-term)',6,81),('linker',82,89),('UBL2 (C-term)',90,163),('C-term tail',164,165)]
def dom(n):
    for nm,lo,hi in DOMAINS:
        if lo<=n<=hi: return nm
    return 'outside'
def rn(l):
    m=re.search(r'(\d+)', l); return int(m.group(1)) if m else 0
rows='\n'.join(f'| {k} | {v:.2f} | {dom(rn(k))} |' for k,v in sorted(occ.items(), key=lambda kv:-kv[1]))
md(f'**MD contact occupancy ({method}):**\n\n| Residue | Occupancy | Domain |\n|---|---|---|\n' + rows)

## Stage 4 — Deterministic adjudication: MD interface vs expected domain


In [ ]:
core = {k:v for k,v in occ.items() if v>=0.5}
in_ubl1 = [k for k in core if 6 <= rn(k) <= 81]
flag = bool(core) and not in_ubl1
verdict = '🚩 **FLAG** — persistent interface is OUTSIDE the expected UBL1 (6–81)' if flag else '✅ consistent with UBL1'
md(f"**Persistent interface (occ ≥ 0.5):** {', '.join(core) or '—'}  \n"
   f"**Inside expected UBL1 (6–81):** {', '.join(in_ubl1) or '**NONE**'}  \n\n### {verdict}")

## Stage 5 — Multi-agent debate (the highlight)

Two agents argue the SAME real facts — one defends the MD C-terminal interface, one
defends the NMR N-terminal expectation — and a judge weighs them, separating *is it
contradictory* (factual) from *which side is right* (unknown).


In [ ]:
if HAVE_KEY:
    from agent.debate import run as run_debate
    run_debate()
    md(open('outputs/debate_report.md').read())
else:
    md('> ⚠️ Set `DEEPSEEK_API_KEY` and re-run this cell to run the live debate.')

## Summary

- **Real:** MD occupancies, PDBe structures, UniProt domains, the contradiction with literature.
- **Not ground truth:** which interface is correct — no experimental complex structure exists.
- **Output:** the model's MD interface (C-terminal) contradicts the NMR-expected UBL1; the
  judge recommends an experimental test rather than declaring a winner.
